<a href="https://colab.research.google.com/github/Gianluca-dot/Progetti/blob/main/Data_Augmentation_Per_La_Sicurezza_Delle_Centrali_Elettriche.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Analisi iniziale

##FASE 1: Setup, Seed, Device & Directory Risultati

In [ ]:
# ==========================================
# CELLA 1: Imports, Seed, Device & Directory
# ==========================================
import os
import gc
import json
import csv
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Setting Seed per Riproducibilità Totale
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizzo device: {device}")

# Utility per Gestione Memoria GPU
def flush_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Creazione Directory Risultati Locale per la Sessione Colab
RESULTS_DIR = "/content/cybereye_results"
os.makedirs(os.path.join(RESULTS_DIR, "baseline"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "augmented"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "synthetic_data"), exist_ok=True)

print(f"Directory risultati pronta in: {RESULTS_DIR}")

##FASE 2: Download OxfordIIITPet & Split Stratificato a 3 Vie

In [ ]:
# ==========================================
# CELLA 2: Dataset Loading & Stratified Split
# ==========================================
from torchvision.datasets import OxfordIIITPet

data_raw_dir = "/content/data_raw"

# Caricamento trainval e test originali
full_trainval_dataset = OxfordIIITPet(root=data_raw_dir, split="trainval", download=True)
full_test_dataset = OxfordIIITPet(root=data_raw_dir, split="test", download=True)

# Estrazione delle etichette da trainval per la stratificazione
trainval_targets = [target for _, target in full_trainval_dataset._images_targets]

# Estrazione del 20% di trainval per creare il Training Ridotto e il Validation Set
indices = list(range(len(full_trainval_dataset)))

# 1. Selezioniamo il subset del 20% dal trainval
reduced_indices, _ = train_test_split(
    indices,
    train_size=0.20,
    stratify=trainval_targets,
    random_state=42
)

reduced_targets = [trainval_targets[i] for i in reduced_indices]

# 2. Dividiamo il subset ridotto in Training (80%) e Validation (20%)
train_idx, val_idx = train_test_split(
    reduced_indices,
    test_size=0.20,
    stratify=reduced_targets,
    random_state=42
)

print(f"Campioni Training Ridotto: {len(train_idx)}")
print(f"Campioni Validation Set:  {len(val_idx)}")
print(f"Campioni Test Set:        {len(full_test_dataset)}")

##FASE 3: Preprocessing, Data Augmentation & Data Loaders

In [ ]:
# ==========================================
# CELLA 3: Transformations & Dataset Wrapper
# ==========================================

# 1. Pipeline di Augmentation Dinamica (solo per il Training)
transform_train = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Preprocessing Deterministico (per Validation e Test)
transform_eval = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Dataset Wrapper per applicare le trasformazioni in modo dinamico
class TransformedSubset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __getitem__(self, idx):
        img, target = self.dataset[self.indices[idx]]
        if self.transform:
            img = self.transform(img)
        return img, target

    def __len__(self):
        return len(self.indices)

# Istanziazione Dataset
train_dataset = TransformedSubset(full_trainval_dataset, train_idx, transform=transform_train)
val_dataset = TransformedSubset(full_trainval_dataset, val_idx, transform=transform_eval)
test_dataset = OxfordIIITPet(root=data_raw_dir, split="test", download=False, transform=transform_eval)

# DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

##FASE 4: Baseline Training con Resume Sicuro ed Early Stopping Gestito

In [ ]:
# ==========================================
# CELLA 4: Modulo di Training con RESUME SICURO ed Early Stopping Integrato
# ==========================================

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_f1):
        if self.best_score is None:
            self.best_score = val_f1
        elif val_f1 < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_f1
            self.counter = 0

    def state_dict(self):
        return {'counter': self.counter, 'best_score': self.best_score}

    def load_state_dict(self, state_dict):
        self.counter = state_dict['counter']
        self.best_score = state_dict['best_score']

def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
    return val_loss, acc, prec, rec, f1

def train_model(model, train_loader, val_loader, save_dir, total_epochs=20):
    criterion = nn.CrossEntropyLoss()
    csv_path = os.path.join(save_dir, "metrics.csv")
    checkpoint_path = os.path.join(save_dir, "checkpoint.pt")

    start_epoch = 1
    best_val_f1 = 0.0

    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

    optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    early_stopping = EarlyStopping(patience=5)

    if os.path.exists(checkpoint_path):
        print(f"--- Trovato Checkpoint esistente in {checkpoint_path}. Ripristino dello stato... ---")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        start_epoch = checkpoint['epoch'] + 1
        best_val_f1 = checkpoint['best_val_f1']

        if start_epoch >= 4:
            for param in model.layer4.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': model.layer4.parameters(), 'lr': 1e-4},
                {'params': model.fc.parameters(), 'lr': 1e-3}
            ], weight_decay=1e-2)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        early_stopping.load_state_dict(checkpoint['early_stopping_state'])
        print(f"--- Ripresa riuscita! Si riparte dall'Epoca {start_epoch} ---")
    else:
        with open(csv_path, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["epoch", "train_loss", "val_loss", "val_acc", "val_prec_macro", "val_rec_macro", "val_f1_macro", "lr"])

    for epoch in range(start_epoch, total_epochs + 1):
        if epoch == 4 and start_epoch < 4:
            print("--- Unfreeze Layer4 per Fine-Tuning Discriminativo ---")
            for param in model.layer4.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': model.layer4.parameters(), 'lr': 1e-4},
                {'params': model.fc.parameters(), 'lr': 1e-3}
            ], weight_decay=1e-2)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate(model, val_loader, criterion)
        current_lr = optimizer.param_groups[0]['lr']

        # 1. Scrittura Metriche
        with open(csv_path, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch, train_loss, val_loss, val_acc, val_prec, val_rec, val_f1, current_lr])

        print(f"Epoca {epoch:02d}/{total_epochs} | Train Loss: {train_loss:.4f} | Val F1-Macro: {val_f1:.4f} | LR: {current_lr}")

        # 2. Aggiornamento Best Model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pt"))

        # 3. Aggiornamento Scheduler ed Early Stopping
        scheduler.step(val_f1)
        early_stopping(val_f1)

        # 4. Salvataggio Checkpoint Sincronizzato
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'early_stopping_state': early_stopping.state_dict(),
            'best_val_f1': best_val_f1
        }
        torch.save(checkpoint, checkpoint_path)

        if early_stopping.early_stop:
            print("Early Stopping attivato.")
            break

# Avvio Baseline
baseline_model = resnet18(weights=ResNet18_Weights.DEFAULT)
baseline_model.fc = nn.Linear(baseline_model.fc.in_features, 37)
baseline_model = baseline_model.to(device)

train_model(baseline_model, train_loader, val_loader, save_dir=os.path.join(RESULTS_DIR, "baseline"))
flush_memory()

##FASE 5: BLIP Captioning Ottimizzato (Solo su 185 Campioni Bilanciati)

In [ ]:
# ==========================================
# CELLA 5: BLIP Captioning Mirato (185 Campioni)
# ==========================================
from collections import defaultdict
from transformers import BlipProcessor, BlipForConditionalGeneration

captions_json_path = os.path.join(RESULTS_DIR, "blip_captions.json")

if os.path.exists(captions_json_path):
    print("--- Caption BLIP già esistenti. Caricamento da disco... ---")
    with open(captions_json_path, "r") as f:
        blip_captions = json.load(f)
else:
    # 1. Selezione preventive dei 5 campioni per classe dal Training Ridotto (37 * 5 = 185 totali)
    class_to_indices = defaultdict(list)
    for idx in train_idx:
        _, label = full_trainval_dataset[idx]
        class_to_indices[label].append(idx)

    selected_indices_with_label = []
    for label, indices_list in class_to_indices.items():
        for idx in indices_list[:5]:
            selected_indices_with_label.append((idx, label))

    print(f"Esecuzione BLIP esclusivamente su {len(selected_indices_with_label)} campioni bilanciati...")

    # 2. Caricamento Modello BLIP
    blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

    blip_captions = []
    class_names = full_trainval_dataset.classes

    for idx, label in tqdm(selected_indices_with_label, desc="Generazione Caption BLIP"):
        raw_img, _ = full_trainval_dataset[idx]
        class_name = class_names[label]

        inputs = blip_processor(raw_img, return_tensors="pt").to(device)
        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=30)
        caption = blip_processor.decode(out[0], skip_special_tokens=True)

        blip_captions.append({
            "original_idx": idx,
            "label": label,
            "class_name": class_name,
            "caption": caption
        })

    with open(captions_json_path, "w") as f:
        json.dump(blip_captions, f, indent=4)

    del blip_model, blip_processor
    flush_memory()

print(f"Disponibili {len(blip_captions)} caption.")

##FASE 6: Espansione Prompt con LLM

In [ ]:
# ==========================================
# CELLA 6: LLM Text Expansion con Vincolo di Classe
# ==========================================
from transformers import AutoModelForCausalLM, AutoTokenizer

prompts_json_path = os.path.join(RESULTS_DIR, "expanded_prompts.json")

if os.path.exists(prompts_json_path):
    print("--- Prompt espansi già esistenti. Caricamento da disco... ---")
    with open(prompts_json_path, "r") as f:
        expanded_prompts = json.load(f)
else:
    model_id = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    llm_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

    expanded_prompts = []
    print(f"Generazione prompt per {len(blip_captions)} campioni...")

    for item in tqdm(blip_captions, desc="LLM Prompt Expansion"):
        system_prompt = (
            f"You are a CCTV surveillance text prompt generator. "
            f"The target breed/animal is explicitly: '{item['class_name']}'. "
            f"Expand the caption into a detailed descriptive prompt. "
            f"STRICT RULE: You MUST retain the exact animal breed '{item['class_name']}'. "
            f"Only vary environment, lighting (e.g. night, rain, fog), camera angle (CCTV style)."
        )
        user_prompt = f"Original Caption: {item['caption']}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(device)

        with torch.no_grad():
            generated_ids = llm_model.generate(**model_inputs, max_new_tokens=60, do_sample=True, temperature=0.7)

        response = tokenizer.batch_decode(generated_ids[:, model_inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

        expanded_prompts.append({
            "label": item["label"],
            "class_name": item["class_name"],
            "prompt": response.strip()
        })

    with open(prompts_json_path, "w") as f:
        json.dump(expanded_prompts, f, indent=4)

    del llm_model, tokenizer
    flush_memory()

print(f"Disponibili {len(expanded_prompts)} prompt per la generazione.")

##FASE 7: Generazione Sintetica con Stable Diffusion

In [ ]:
# ==========================================
# CELLA 7: Stable Diffusion Generation con Resume
# ==========================================
from diffusers import StableDiffusionPipeline

synth_dir = os.path.join(RESULTS_DIR, "synthetic_data")
synthetic_records = []

existing_files = [f for f in os.listdir(synth_dir) if f.startswith("synth_") and f.endswith(".png")]

if len(existing_files) == len(expanded_prompts):
    print("--- Immagini sintetiche già completamente generate! ---")
    for i, record in enumerate(expanded_prompts):
        img_filename = f"synth_{i:04d}_class_{record['label']}.png"
        synthetic_records.append({
            "image_path": os.path.join(synth_dir, img_filename),
            "label": record["label"]
        })
else:
    print("Caricamento Stable Diffusion in GPU...")
    sd_pipeline = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16
    ).to(device)

    negative_prompt = "blurry, low quality, wrong species, distorted, deformed, bad anatomy"

    for i, record in enumerate(tqdm(expanded_prompts, desc="Generazione Immagini Sintetiche")):
        img_filename = f"synth_{i:04d}_class_{record['label']}.png"
        img_path = os.path.join(synth_dir, img_filename)

        if not os.path.exists(img_path):
            image = sd_pipeline(
                prompt=record["prompt"],
                negative_prompt=negative_prompt,
                num_inference_steps=25,
                guidance_scale=7.5
            ).images[0]
            image.save(img_path)

        synthetic_records.append({
            "image_path": img_path,
            "label": record["label"]
        })

    del sd_pipeline
    flush_memory()

print(f"Salvate e registrate {len(synthetic_records)} immagini sintetiche in {synth_dir}")

##FASE 8: Augmented Training (Original + Synthetic)

In [ ]:
# ==========================================
# CELLA 8: Addestramento Modello Augmented
# ==========================================

class SyntheticDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = Image.open(rec["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, rec["label"]

    def __len__(self):
        return len(self.records)

synthetic_dataset = SyntheticDataset(synthetic_records, transform=transform_train)
augmented_train_dataset = ConcatDataset([train_dataset, synthetic_dataset])
augmented_train_loader = DataLoader(augmented_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"Dimensione totale Dataset Augmented: {len(augmented_train_dataset)}")

augmented_model = resnet18(weights=ResNet18_Weights.DEFAULT)
augmented_model.fc = nn.Linear(augmented_model.fc.in_features, 37)
augmented_model = augmented_model.to(device)

print("Inizio Addestramento Modello Augmented...")
train_model(augmented_model, augmented_train_loader, val_loader, save_dir=os.path.join(RESULTS_DIR, "augmented"))
flush_memory()

##FASE 9: Valutazione Finale Simmetrica sul Test Set & Salvataggio Matrici di Confusione

In [ ]:
# ==========================================
# CELLA 9: Final Test Evaluation & Confusion Matrix
# ==========================================

def evaluate_test_final(model_path, dataloader):
    model = resnet18()
    model.fc = nn.Linear(model.fc.in_features, 37)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    return {"accuracy": acc, "precision_macro": prec, "recall_macro": rec, "f1_macro": f1}, cm.tolist()

# Valutazione Baseline
baseline_best_path = os.path.join(RESULTS_DIR, "baseline", "best_model.pt")
baseline_test_metrics, baseline_cm = evaluate_test_final(baseline_best_path, test_loader)

# Valutazione Augmented
augmented_best_path = os.path.join(RESULTS_DIR, "augmented", "best_model.pt")
augmented_test_metrics, augmented_cm = evaluate_test_final(augmented_best_path, test_loader)

# 1. Salvataggio Separato delle Matrici di Confusione
with open(os.path.join(RESULTS_DIR, "baseline", "confusion_matrix.json"), "w") as f:
    json.dump(baseline_cm, f)

with open(os.path.join(RESULTS_DIR, "augmented", "confusion_matrix.json"), "w") as f:
    json.dump(augmented_cm, f)

# 2. Salvataggio Metriche Finali
final_metrics = {
    "baseline": baseline_test_metrics,
    "augmented": augmented_test_metrics
}

with open(os.path.join(RESULTS_DIR, "final_test_comparison.json"), "w") as f:
    json.dump(final_metrics, f, indent=4)

print("=== RISULTATI FINALI SUL TEST SET ===")
print(f"Baseline  | Accuracy: {baseline_test_metrics['accuracy']:.4f} | F1-Macro: {baseline_test_metrics['f1_macro']:.4f}")
print(f"Augmented | Accuracy: {augmented_test_metrics['accuracy']:.4f} | F1-Macro: {augmented_test_metrics['f1_macro']:.4f}")
print(f"\nMatrici di confusione salvate rispettivamente in:\n- {os.path.join(RESULTS_DIR, 'baseline', 'confusion_matrix.json')}\n- {os.path.join(RESULTS_DIR, 'augmented', 'confusion_matrix.json')}")

##Analisi dei risultati e Conclusioni finali